In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import harmonypy as hm
import scanpy.external as sce
import squidpy as sq
import spatialdata as sd
import spatialdata_plot as sdp
from spatialdata.models.models import ShapesModel
import os
from matplotlib.colors import TwoSlopeNorm, Normalize
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon

# QC

In [ ]:
tma6_og = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/xenium_mistic/tma6_xenium_clean_mistic.h5ad")

In [ ]:
tma6_og

In [ ]:
## add the meta info back
tma6l_meta = pd.read_csv('/homevol/kk/analysis/OVA_TMA/Old/TMA5_TMA6/tma6l_og_metadata.csv')
tma6r_meta = pd.read_csv('/homevol/kk/analysis/OVA_TMA/Old/TMA5_TMA6/tma6r_og_metadata.csv')
tma6_meta = pd.concat([tma6l_meta, tma6r_meta], axis=0, ignore_index=True)
tma6_meta = tma6_meta.set_index('cell_id')


In [ ]:
# Ensure ordering matches
tma6_meta = tma6_meta.loc[tma6_og.obs.index]

# # Overwrite obs
tma6_og.obs[['arrayID', 'tma', 'PatientID', 'Diagnosis']] = tma6_meta[['arrayID', 'tma', 'PatientID', 'Diagnosis']]


In [ ]:
## drop out unassigned cells or TMAs

keep = tma6_og.obs["arrayID"] != "unassigned"

if "PatientID" in tma6_og.obs.columns:
    keep &= tma6_og.obs["PatientID"].notna()

if "Diagnosis" in tma6_og.obs.columns:
    keep &= tma6_og.obs["Diagnosis"].notna()

tma6_og = tma6_og[keep].copy()


In [ ]:
sc.pp.calculate_qc_metrics(tma6_og, inplace=True, log1p=True, percent_top=None)


In [ ]:
tma6_og

In [ ]:
## Filter out low quality cells
tma6 = tma6_og[
    (tma6_og.obs["total_counts"] >= 10) &
    (tma6_og.obs["n_genes_by_counts"] > 0)
].copy()

In [ ]:
tma6

In [ ]:
## check TMA core wise cell count distribution

# Count cells per array
array_counts = (
    tma6.obs.groupby("arrayID")
    .size()
    .reset_index(name="cell_count")
)
array_meta = (
    tma6.obs.groupby("arrayID")[["Diagnosis", "PatientID"]]
    .first()
    .reset_index()
)

array_counts = (
    array_counts
    .merge(array_meta, on="arrayID")
    .sort_values("cell_count", ascending=True)
)

In [ ]:
## Filter out TMA cores with less than 100 cells in total

valid_arrays = array_counts.loc[array_counts["cell_count"] >= 100, "arrayID"]

# Subset AnnData to only those arrays
tma6 = tma6[tma6.obs["arrayID"].isin(valid_arrays)].copy()

tma6.obs.head()

In [ ]:
tma6_og.obs['PatientID'].nunique()

In [ ]:
tma6

In [ ]:
tma6.X[1:5, 1:5].toarray()

In [ ]:
tma6.layers["raw_counts"] = tma6.X.copy()

In [ ]:
sc.pp.normalize_total(tma6)
sc.pp.log1p(tma6)
sc.tl.pca(tma6)
sc.pp.neighbors(tma6, n_pcs= 10)
sc.tl.umap(tma6)
sc.tl.leiden(tma6, key_added="leiden_res_0.1", resolution=0.1)


In [ ]:
sc.pl.umap(tma6, color="leiden_res_0.1", 
            legend_loc="on data",
            legend_fontsize=18,
            legend_fontoutline=2
           )

In [ ]:
sc.tl.rank_genes_groups(tma6, groupby= "leiden_res_0.1", method='wilcoxon', key_added="rank_genes_0.1")

In [ ]:
sc.pl.rank_genes_groups_dotplot(tma6, 
                                groupby="leiden_res_0.1", 
                                key="rank_genes_0.1",
                                n_genes=5)

In [ ]:
sc.pl.rank_genes_groups_dotplot(tma6, 
                                groupby="leiden_res_0.1", 
                                key="rank_genes_0.1",
                                n_genes=5,
                                values_to_plot="logfoldchanges",
                                cmap="bwr",
                                vcenter=0,
                                vmin=-5,
                                vmax=5,
                                min_logfoldchange=2)

## Supervised annotation with Decoupler (ULM)

In [ ]:
markers = pd.read_csv(
    "/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v2/panel_annot_broad_labels_for_annot_tools_AF_v2.csv",
    encoding="cp1252"
)

In [ ]:
## marker list prep

markers = markers.drop_duplicates()

markers = markers.rename(columns={"Adjusted_group": "source", "gene": "target"})

markers = markers.loc[:, ~markers.columns.duplicated()]

markers = markers[["source", "target"]]

In [ ]:
dc.run_ulm(
        mat=tma6,
        net=markers,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6.obs["decoupler"] = tma6.obsm["ulm_estimate"].idxmax(axis=1)


In [ ]:
tma6

In [ ]:
### Filter mismatched cells between coarse leiden and decoupler

dec = tma6.obs["decoupler"].astype(str)
cl  = tma6.obs["leiden_res_0.1"].astype(str)

malignant_only = {"Tumour"}
mismatch = np.full(dec.shape, False)


mismatch_cl0 = (cl == "0") & (~dec.isin(["Tumour", "Proliferating"]))

mismatch_cl12 = cl.isin(["1", "2"]) & (dec.isin(malignant_only))

mismatch = mismatch_cl0 | mismatch_cl12

tma6.obs["malignancy_mismatch"] = np.where(mismatch, "Mismatch", "Match")


In [ ]:
tma6.obs["malignancy_mismatch"].value_counts()

In [ ]:
pd.crosstab(
    tma6.obs["decoupler"],
    tma6.obs["malignancy_mismatch"]
)


In [ ]:
## filter out mismatched cells

tma6_filt = tma6[tma6.obs["malignancy_mismatch"] != "Mismatch"].copy()

In [ ]:
tma6_filt

In [ ]:
tma6_filt.write("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt.h5ad")

# Sub-annotation of decoupler primary lineage types into specific cell sub-populations

In [ ]:
tma6 = sc.read("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt.h5ad")
tma6

In [ ]:
# Get all fine labels
celltypes = (
    markers["fine_label_AF"]
    .dropna()
    .unique()
    .tolist()
)

celltypes_ordered = [ct for ct in celltypes if ct != "-"]
if "-" in celltypes:
    celltypes_ordered.append("-")

celltype_markers = {
    ct: sorted(
        markers.loc[markers["fine_label_AF"] == ct, "gene"]
        .unique()
        .tolist()
    )
    for ct in celltypes_ordered
}


## T-cells

In [ ]:
tma6_t = tma6[tma6.obs["decoupler"] == "T cells"].copy()
tma6_t

In [ ]:
tma6_t.X = tma6_t.layers['raw_counts'].copy()

In [ ]:
tma6_t.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_t)
sc.pp.log1p(tma6_t)
sc.tl.pca(tma6_t)

In [ ]:
sc.pl.pca_variance_ratio(tma6_t, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_t)
sc.tl.umap(tma6_t)
sc.tl.leiden(tma6_t)

In [ ]:
tma6_t

In [ ]:
markers_t = markers[markers["Adjusted_group"] == "T cells"].copy()
markers_t = markers_t[markers_t["fine_label_AF"] != "-"]
markers_t = markers_t.drop_duplicates()
markers_t = markers_t.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_t = markers_t.loc[:, ~markers_t.columns.duplicated()]
markers_t = markers_t[["source", "target"]]
markers_t

In [ ]:
dc.run_ulm(
        mat=tma6_t,
        net=markers_t,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_t.obs["decoupler_t"] = tma6_t.obsm["ulm_estimate"].idxmax(axis=1)


## B-cells

In [ ]:
tma6_b = tma6[tma6.obs["decoupler"] == "B_plasma cells"].copy()
tma6_b

In [ ]:
tma6_b.X = tma6_b.layers['raw_counts'].copy()

In [ ]:
tma6_b.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_b)
sc.pp.log1p(tma6_b)
sc.tl.pca(tma6_b)

In [ ]:
sc.pl.pca_variance_ratio(tma6_b, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_b)
sc.tl.umap(tma6_b)
sc.tl.leiden(tma6_b)

In [ ]:
markers_b = markers[markers["Adjusted_group"] == "B_plasma cells"].copy()
markers_b = markers_b[markers_b["fine_label_AF"] != "-"]
markers_b = markers_b.drop_duplicates()
markers_b = markers_b.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_b = markers_b.loc[:, ~markers_b.columns.duplicated()]
markers_b = markers_b[["source", "target"]]
markers_b

In [ ]:
dc.run_ulm(
        mat=tma6_b,
        net=markers_b,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_b.obs["decoupler_b"] = tma6_b.obsm["ulm_estimate"].idxmax(axis=1)


## Mono_macrophages

In [ ]:
tma6_mac = tma6[tma6.obs["decoupler"] == "Monocyte_Macrophages"].copy()
tma6_mac

In [ ]:
tma6_mac.X = tma6_mac.layers['raw_counts'].copy()

In [ ]:
tma6_mac.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_mac)
sc.pp.log1p(tma6_mac)
sc.tl.pca(tma6_mac)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_mac, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_mac)
sc.tl.umap(tma6_mac)
sc.tl.leiden(tma6_mac)

In [ ]:
tma6_mac

In [ ]:
markers_mac = markers[markers["Adjusted_group"] == "Monocyte_Macrophages"].copy()
markers_mac = markers_mac[markers_mac["fine_label_AF"] != "-"]
markers_mac = markers_mac.drop_duplicates()
markers_mac = markers_mac.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_mac = markers_mac.loc[:, ~markers_mac.columns.duplicated()]
markers_mac = markers_mac[["source", "target"]]
markers_mac

In [ ]:
dc.run_ulm(
        mat=tma6_mac,
        net=markers_mac,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_mac.obs["decoupler_mac"] = tma6_mac.obsm["ulm_estimate"].idxmax(axis=1)


## Fibroblasts

In [ ]:
tma6_fib = tma6[tma6.obs["decoupler"] == "Fibroblast"].copy()
tma6_fib

In [ ]:
tma6_fib.X = tma6_fib.layers['raw_counts'].copy()

In [ ]:
tma6_fib.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_fib)
sc.pp.log1p(tma6_fib)
sc.tl.pca(tma6_fib)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_fib, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_fib)
sc.tl.umap(tma6_fib)
sc.tl.leiden(tma6_fib)

In [ ]:
tma6_fib

In [ ]:
markers_fib = markers[markers["Adjusted_group"] == "Fibroblast"].copy()
markers_fib = markers_fib[markers_fib["fine_label_AF"] != "-"]
markers_fib = markers_fib.drop_duplicates()
markers_fib = markers_fib.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_fib = markers_fib.loc[:, ~markers_fib.columns.duplicated()]
markers_fib = markers_fib[["source", "target"]]
markers_fib

In [ ]:
dc.run_ulm(
        mat=tma6_fib,
        net=markers_fib,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_fib.obs["decoupler_fib"] = tma6_fib.obsm["ulm_estimate"].idxmax(axis=1)


## Endothelial

In [ ]:
sc.set_figure_params(fontsize=14) 
plt.rcParams['patch.edgecolor'] = 'black'
sc.set_figure_params(figsize=(8, 6)) 

In [ ]:
tma6_endo = tma6[tma6.obs["decoupler"] == "Endothelial"].copy()
tma6_endo

In [ ]:
tma6_endo.X = tma6_endo.layers['raw_counts'].copy()

In [ ]:
tma6_endo.X[1:5, 1:5].toarray()

In [ ]:
sc.pp.normalize_total(tma6_endo)
sc.pp.log1p(tma6_endo)
sc.tl.pca(tma6_endo)

In [ ]:
# Inspect variance explained
sc.pl.pca_variance_ratio(tma6_endo, log=True, n_pcs=50)


In [ ]:
sc.pp.neighbors(tma6_endo)
sc.tl.umap(tma6_endo)
sc.tl.leiden(tma6_endo)

In [ ]:
tma6_endo

In [ ]:
markers_endo = markers[markers["Adjusted_group"] == "Endothelial"].copy()
markers_endo = markers_endo[markers_endo["fine_label_AF"] != "-"]
markers_endo = markers_endo.drop_duplicates()
markers_endo = markers_endo.rename(columns={"fine_label_AF": "source", "gene": "target"})
markers_endo = markers_endo.loc[:, ~markers_endo.columns.duplicated()]
markers_endo = markers_endo[["source", "target"]]
markers_endo

In [ ]:
dc.run_ulm(
        mat=tma6_endo,
        net=markers_endo,
        weight = None, 
        min_n= 1,
        verbose= True,
        use_raw= False
    )

# assign top-scoring cell type
tma6_endo.obs["decoupler_endo"] = tma6_endo.obsm["ulm_estimate"].idxmax(axis=1)


## Incorporate the annotations to the main anndata

In [ ]:
fine_tables = [
    tma6_t.obs[["cell_id", "decoupler_t"]].rename(columns={"decoupler_t": "decoupler_fine"}),
    tma6_fib.obs[["cell_id", "decoupler_fib"]].rename(columns={"decoupler_fib": "decoupler_fine"}),
    tma6_endo.obs[["cell_id", "decoupler_endo"]].rename(columns={"decoupler_endo": "decoupler_fine"}),
    tma6_mac.obs[["cell_id", "decoupler_mac"]].rename(columns={"decoupler_mac": "decoupler_fine"}),
    tma6_b.obs[["cell_id", "decoupler_b"]].rename(columns={"decoupler_b": "decoupler_fine"}),
]

fine_df = pd.concat(fine_tables, axis=0)
fine_df = fine_df.set_index("cell_id")
fine_df.head()

In [ ]:
tma6.obs["decoupler_fine"] = tma6.obs["decoupler"].astype("category")

new_cats = pd.Index(fine_df["decoupler_fine"].unique())
tma6.obs["decoupler_fine"] = tma6.obs["decoupler_fine"].cat.add_categories(new_cats)

tma6.obs.loc[fine_df.index, "decoupler_fine"] = fine_df["decoupler_fine"]
tma6.obs["decoupler_fine"] = tma6.obs["decoupler_fine"].cat.remove_unused_categories()

In [ ]:
tma6.write_h5ad("/homevol/kk/analysis/OVA_TMA/TMA6_notebook/adjustments_mis_assigned_transcripts/supervised_annotation/v3/tma6_xen_mist_coarse_decoupler_filt_fine.h5ad")